# 14 · Variance Swaps & Quanto Options

Variance swaps reference realized variance plus implied vol surfaces, while quanto options couple foreign equity with domestic payouts via correlation adjustments.

### Learning Objectives
- Assemble realized price series and implied volatility surfaces for variance swaps
- Configure variance swap parameters (strike variance, observation frequency, day-count)
- Build quanto options with equity + FX inputs and correlation adjustments

In [ ]:
from datetime import date

from finstack import Money
from finstack.core.currency import USD, EUR
from finstack.core.dates.daycount import DayCount
from finstack.core.dates.schedule import Frequency
from finstack.core.market_data.context import MarketContext
from finstack.core.market_data.scalars import MarketScalar, ScalarTimeSeries, SeriesInterpolation
from finstack.core.market_data.surfaces import VolSurface
from finstack.core.market_data.term_structures import DiscountCurve
from finstack.valuations.instruments import VarianceSwap, QuantoOption
from finstack.valuations.pricer import create_standard_registry

as_of = date(2024, 7, 1)
market = MarketContext()
market.insert_discount(
    DiscountCurve(
        "USD-OIS",
        as_of,
        [(0.0, 1.0), (0.5, 0.9980), (1.0, 0.9960), (3.0, 0.9820)],
    )
)
market.insert_discount(
    DiscountCurve(
        "EUR-ESTR",
        as_of,
        [(0.0, 1.0), (0.5, 0.9985), (1.0, 0.9970)],
    )
)
observations = [
    (date(2023, 9, 29), 4200.0),
    (date(2023, 12, 29), 4305.0),
    (date(2024, 3, 28), 4380.0),
    (date(2024, 6, 28), 4450.0),
]
market.insert_series(
    ScalarTimeSeries(
        "SPX-LEVELS",
        observations,
        currency=USD,
        interpolation=SeriesInterpolation.LINEAR,
    )
)
market.insert_price("SPX", MarketScalar.price(Money(observations[-1][1], USD)))
market.insert_surface(
    VolSurface(
        "SPX-VOL",
        expiries=[0.25, 0.5, 1.0],
        strikes=[3500.0, 4000.0, 4500.0],
        grid=[
            [0.22, 0.21, 0.20],
            [0.24, 0.22, 0.21],
            [0.26, 0.24, 0.22],
        ],
    )
)
market.insert_surface(
    VolSurface(
        "SAP-VOL",
        expiries=[0.5, 1.0],
        strikes=[100.0, 120.0, 140.0, 160.0, 180.0],
        grid=[
            [0.30, 0.27, 0.25, 0.27, 0.30],
            [0.32, 0.29, 0.27, 0.29, 0.32],
        ],
    )
)
market.insert_surface(
    VolSurface(
        "EURUSD-VOL",
        expiries=[0.5, 1.0],
        strikes=[1.00, 1.05, 1.10, 1.15, 1.20],
        grid=[
            [0.08, 0.07, 0.06, 0.07, 0.08],
            [0.09, 0.08, 0.07, 0.08, 0.09],
        ],
    )
)
market.insert_price("SAP", MarketScalar.price(Money(140.0, EUR)))
market.insert_price("SAP.DIV", MarketScalar.unitless(0.02))
market.insert_price("EURUSD", MarketScalar.unitless(1.10))
registry = create_standard_registry()


## 1. Variance Swap
Supply strike variance, observation window, and realized series ID (here inferred via `observation_frequency`).

In [ ]:
variance_swap = VarianceSwap.create(
    "SPX-VAR-SWAP",
    underlying_id="SPX",
    notional=Money(1_000_000, USD),
    strike_variance=0.04,
    start_date=date(2024, 1, 1),
    maturity=date(2024, 12, 31),
    discount_curve="USD-OIS",
    observation_frequency=Frequency.QUARTERLY,
    realized_method=None,
    side="receive",
    day_count=DayCount.ACT_365F,
)
var_res = registry.price_with_metrics(variance_swap, "discounting", market, ["variance_vega", "variance_expected"], as_of=as_of)
print("Variance swap PV:", var_res.value)
print("Expected variance:", var_res.measures.get("variance_expected"))
print("Vega:", var_res.measures.get("variance_vega"))


## 2. Quanto Option
Quanto options pay in domestic currency but reference a foreign underlying. Provide both equity and FX vol references plus correlation.

In [ ]:
quanto_call = QuantoOption.builder(
    instrument_id="QUANTO-CALL",
    ticker="SAP",
    equity_strike=140.0,
    option_type="call",
    expiry=date(2025, 12, 31),
    notional=Money(10000.0, USD),
    domestic_currency=USD,
    foreign_currency=EUR,
    correlation=0.3,
    discount_curve="USD-OIS",
    foreign_discount_curve="EUR-ESTR",
    spot_id="SAP",
    vol_surface="SAP-VOL",
    div_yield_id="SAP.DIV",
    fx_rate_id="EURUSD",
    fx_vol_id="EURUSD-VOL",
)
quanto_res = registry.price(quanto_call, "monte_carlo_gbm", market, as_of=as_of)
print("Quanto call PV:", quanto_res.value)


## Summary
- Variance swaps need realized history (`ScalarTimeSeries`) plus implied vol surfaces for expectation calculations.
- Quanto options mix equity and FX data, including correlation assumptions, to deliver domestic currency exposure to foreign equities.